# Survivor Data Set: An Exploration
## Project Overview

We seek to understand the underlying themes in the reality TV gameshow "Survivor". The show is currently airing it's 50th season, but the gameplay we see in the later seasons is very different from the early seasons of the show. Survivor is a gameshow about physical challenges and mental strength, but fundamentally it is a show about relationships with others. In a game where the goal is to "Outwit, Outplay, Outlast", what does this actually translate to? In order to win you need the votes of the jury, which is made up of the players you had a hand in voting off of the tribe.

The thing that makes Survivor so interesting is how different every season is, and how the game is always evolving. In early seasons, loyalty and morals were valued, and the gameplay was relatively simple. As time went on, players got more conniving, and blindsides and backstabbing became the norm. But what is most interesting is that as the gameplay changed, so did the mentality of the players. Playing fair isn't enough to win anymore. 

Through this data we hope to find answers to some of these questions:
- What does it take Outwit, Outplay, and Outlast?
- What decisions to winners make? 
- What decisions do losers make?
- How have the decisions made by winners and losers changed as the game has evolved?
- Are there trends that predict performance?
- Are there qualities that predict performance?
- Have there been changes in these trends/qualities over time?

## Data Description and Source
The original survivoR dataset is in R, [original data set](https://github.com/doehm/survivoR/tree/master/data). We will be using a modified version of this dataset that has been converted to CSV files found here: [Survivor Data Set](https://github.com/rfordatascience/tidytuesday/tree/master/data/2021/2021-06-01)

The original dataset has 23 R data files, while the dataset that's been converted to CSV only has 5 of these. 

The 5 csv files are:
- Summary
- Challanges
- Castaways
- Viewers
- Jury Votes


# Initial CSV Data
The sections below walk through each CSV in the same order: **Initial CSV exploration**, **Cleaning needs**, **Methods**, and **Results**.


## Setup: imports and loading

Import **pandas**, **matplotlib** / **seaborn** for plots, **IPython.display** for styled tables, then load all tables from the TidyTuesday CSV mirror.


In [ ]:
# Imports for the project
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

In [ ]:
# Each table is available as a CSV from the TidyTuesday GitHub mirror
base_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-06-01/"

# Load the main tables
summary_df     = pd.read_csv(base_url + "summary.csv")
challenges_df  = pd.read_csv(base_url + "challenges.csv")
castaways_df   = pd.read_csv(base_url + "castaways.csv")
viewers_df     = pd.read_csv(base_url + "viewers.csv")
jury_votes_df  = pd.read_csv(base_url + "jury_votes.csv")

## Table: Summary


### Initial CSV exploration

#### Relevance:
- Primary data set for information on seasons, winners, and dates.
- Useful for determining winners and trends over time.

#### Size:
- 40 rows
- 19 columns
- 6.1 KB

#### Column descriptions (English):
- **Season Name**: The name of the season
- **Season**: The season number
- **Location**: The geographical location of the show
- **Country**: The country where the season takes place
- **Tribe Setup**: How players are divided into tribes (teams)
- **Full Name**: The name of the player
- **Winner**: The winner of the season
- **Runner Ups**: Second place
- **Final Vote**: The vote split for the winner
- **Time Slot**: Day and time of week when episodes aired
- **Premiered**: Date when the season premiered
- **Ended**: Date when the season ended
- **Filming Strated**: Date when filming started (note: this spelling matches the CSV column label)
- **Filming Ended**: Date when filming ended
- **Viewers Finale**: Number of viewers for the final episode
- **Viewers Reunion**: Number of viewers for the reunion (contestant retrospective)
- **Viewers Mean**: Average viewership for episodes
- **Rank**: Viewer final ranking of the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
summary_df.info()
summary_df.head()

### Cleaning

All steps use `clean_summary_df`, created with `summary_df.copy()` in the first cleaning code cell, so `summary_df` stays the untouched load. From raw inspection, we fix:

- `premiered`, `ended`, `filming_started`, and `filming_ended`: strings → datetimes
- `timeslot`: split into `weekday` and `air_time`
- `viewers_mean` and `rank`: median fill for `viewers_mean` and missing `rank` kept with `has_rank` for training filters


### Methods

The following code cells implement the cleaning steps on `clean_summary_df`.


In [ ]:
# Working copy for cleaning; summary_df remains the raw CSV load
clean_summary_df = summary_df.copy()

In [ ]:
# View the timeslot column entries
clean_summary_df['timeslot'].value_counts()

In [ ]:
# Extract weekday and time text from timeslot
parts = clean_summary_df["timeslot"].str.extract(r"(\w+)\s+(.+)")
clean_summary_df["weekday"] = parts[0]
clean_summary_df["air_time_raw"] = parts[1]

# Parse time text into a time value
clean_summary_df["air_time"] = pd.to_datetime(
    clean_summary_df["air_time_raw"].str.strip().str.lower(),
    format="%I:%M %p",
    errors="coerce"
).dt.time

# Drop the timeslot column and airtime_raw column
clean_summary_df = clean_summary_df.drop(columns=["timeslot", "air_time_raw"])

clean_summary_df.info()

In [ ]:
# Convert the premiered, ended, filming_started, and filming_ended columns to datetime
clean_summary_df['premiered'] = pd.to_datetime(clean_summary_df['premiered'])
clean_summary_df['ended'] = pd.to_datetime(clean_summary_df['ended'])
clean_summary_df['filming_started'] = pd.to_datetime(clean_summary_df['filming_started'])
clean_summary_df['filming_ended'] = pd.to_datetime(clean_summary_df['filming_ended'])

# Show the updated dataframe
clean_summary_df.info()

In [ ]:
# View the null values in viewers_mean and rank
clean_summary_df[clean_summary_df['viewers_mean'].isna() | clean_summary_df['rank'].isna()]

`rank` and `viewers_mean` are missing in the same two rows. We keep every season, so `viewers_mean` is filled with the median of non-missing values, `rank` stays NaN where unknown, and `has_rank` is True only when `rank` is present so rows without a label can be excluded when training a model that predicts `rank`.

In [ ]:
# Median impute viewers_mean; keep missing rank and flag rows usable as labeled training data
viewers_median = clean_summary_df["viewers_mean"].median()
clean_summary_df["viewers_mean"] = clean_summary_df["viewers_mean"].fillna(
    viewers_median
)
clean_summary_df["has_rank"] = clean_summary_df["rank"].notna()

clean_summary_df.info()

### Results

**Summary:** Summary Data Frame

`clean_summary_df` keeps all 40 seasons with consistent dtypes and `weekday` / `air_time` from `timeslot`. After cleaning, `viewers_mean` has no nulls, from median imputation; `rank` is still null for the two incomplete seasons, so `has_rank` marks rows to include when `rank` is a target.


## Table: Challenges


### Initial CSV exploration

#### Relevance:
- Examines outcomes when contestants and tribes compete; these outcomes influence decisions when tribes vote contestants out.

#### Size:
- 5023 rows
- 8 columns
- 314.1 KB

#### Column descriptions (English):
- **season_name**: The season's name
- **season**: The season number
- **episode**: The episode number within the season
- **title**: The title of the episode
- **day**: The running day count since the game began
- **challenge_type**: Reward (a prize) or immunity (protection from being voted out)
- **winners**: The name of the contestant who won the challenge
- **winning_tribe**: The name of the tribe that won the challenge

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
print(challenges_df.info())
print(challenges_df.head())

# Challenge nulls before cleaning
challenge_nulls_before = challenges_df[["winners", "winning_tribe"]].isna().sum()
print("Nulls before cleaning:")
print(challenge_nulls_before)

### Cleaning Needs / Methods

The following code cell houses all the challenge-row cleaning rules for `challenges_df`.


In [ ]:
# Clean Version
challenges_cleaned_df = challenges_df.copy()

# Dropping rows that are placeholder / non-challenge rows
drop_mask = (
    # Survivor: Island of the Idols, episode 12
    ((challenges_cleaned_df["season"] == 39) &
     (challenges_cleaned_df["episode"] == 12) &
     (challenges_cleaned_df["day"] == 36) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna())) |

    # Survivor: David vs. Goliath, episode 4
    ((challenges_cleaned_df["season"] == 37) &
     (challenges_cleaned_df["episode"] == 4) &
     (challenges_cleaned_df["day"] == 10) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~drop_mask].copy()

# Filling in rows where there truly was no immunity winner
no_winner_cases = [
    (32, 13),  # Kaoh Rong (Joe evacuated)
    (24, 6),   # One World (Colton evacuated)
    (21, 12),  # Nicaragua (NaOnka and Kelly quit)
    (19, 6),   # Samoa (Russell Swan evacuated)
    (12, 11),  # Panama (Bruce evacuated)
    (8, 3),    # All-Stars (Jenna quit)
    (8, 6),    # All-Stars (Sue quit)
    (2, 6)     # Australian Outback (Michael evacuated)
]

for season_num, episode_num in no_winner_cases:
    mask = (
        (challenges_cleaned_df["season"] == season_num) &
        (challenges_cleaned_df["episode"] == episode_num) &
        (challenges_cleaned_df["challenge_type"] == "immunity") &
        (challenges_cleaned_df["winners"].isna())
    )

    challenges_cleaned_df.loc[mask, "winners"] = "No challenge winner"
    challenges_cleaned_df.loc[mask, "winning_tribe"] = "Not applicable"

# If a winner exists but winning_tribe is missing, then tribe winner does not apply
tribe_not_applicable_mask = (
    challenges_cleaned_df["winners"].notna() &
    challenges_cleaned_df["winning_tribe"].isna()
)

challenges_cleaned_df.loc[tribe_not_applicable_mask, "winning_tribe"] = "Not applicable"

# Fixing Survivor: Blood vs. Water, episode 1
# Galang won the combined immunity/reward challenge, so restore the missing immunity winners
bvw_bad_immunity_rows = (
    (challenges_cleaned_df["season"] == 27) &
    (challenges_cleaned_df["episode"] == 1) &
    (challenges_cleaned_df["day"] == 1) &
    (challenges_cleaned_df["challenge_type"] == "immunity") &
    (challenges_cleaned_df["winners"].isna())
)

challenges_cleaned_df = challenges_cleaned_df.loc[~bvw_bad_immunity_rows].copy()

galang_members = [
    "Aras",
    "Colton",
    "Gervase",
    "Kat",
    "Laura B.",
    "Laura M.",
    "Monica",
    "Tina",
    "Tyson"
]

bvw_immunity_rows = pd.DataFrame(
    [
        {
            "season_name": "Survivor: Blood vs. Water",
            "season": 27,
            "episode": 1,
            "title": "Blood Is Thicker Than Anything",
            "day": 1,
            "challenge_type": "immunity",
            "winners": member,
            "winning_tribe": "Galang"
        }
        for member in galang_members
    ]
)

challenges_cleaned_df = pd.concat(
    [challenges_cleaned_df, bvw_immunity_rows],
    ignore_index=True
)

# Drop the last 3 blank reward placeholder rows
final_placeholder_rows = (
    ((challenges_cleaned_df["season"] == 22) &
     (challenges_cleaned_df["episode"] == 14) &
     (challenges_cleaned_df["day"] == 38) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 16) &
     (challenges_cleaned_df["episode"] == 1) &
     (challenges_cleaned_df["day"] == 3) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 13) &
     (challenges_cleaned_df["episode"] == 6) &
     (challenges_cleaned_df["day"] == 15) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~final_placeholder_rows].copy()

# Sort the cleaned df so the column names make sense
challenges_cleaned_df = challenges_cleaned_df.sort_values(
    by=["season", "episode", "day", "challenge_type", "winners"]
).reset_index(drop=True)

# Challenge nulls after cleaning
challenge_nulls_after = challenges_cleaned_df[["winners", "winning_tribe"]].isna().sum()
print("\nChallenge nulls after cleaning:")
print(challenge_nulls_after)

print(challenges_cleaned_df.info())
print(challenges_cleaned_df.head())

### Results

See the printed null counts and `challenges_cleaned_df.info()` / `head()` output above for this run.


## Table: Castaways


### Initial CSV exploration

#### Relevance:
- Personal data on each contestant and their performance.
- Useful for analyzing how traits relate to outcomes.

#### Size:
- 744 rows
- 18 columns
- 104.8 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **full_name**: The contestant's full name
- **castaway**: The castaway's (contestant's) first name
- **age**: The contestant's age
- **city**: The city the contestant is from
- **state**: The state the contestant is from
- **personality_type**: Their personality description
- **day**: The day of the season (running count)
- **order**: Finish order for the season (larger values mean the contestant lasted longer)
- **result**: When they were voted out (string description of the order column)
- **jury_status**: If and when the player made the jury
- **original_tribe**: The tribe they started on
- **swapped_tribe**: The tribe they swapped to
- **swapped_tribe2**: The tribe they swapped to a second time
- **merged_tribe**: The merged tribe name
- **total_votes_received**: Number of votes cast against the contestant
- **immunity_idols_won**: Number of immunity idols won by the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
castaways_df.info()
castaways_df.head()

#### Raw inspection (continued)

The following information is used to determine the basic information about the dataset that is needed, and explained at the top of this section.


In [ ]:
# Shape and column names
print("SHAPE")
print(castaways_df.shape)

print("\nCOLUMNS")
print(castaways_df.columns.tolist())

# Data types
print("\nDATA TYPES")
print(castaways_df.dtypes)

# Missing values
print("\nMISSING VALUES")
print(castaways_df.isnull().sum())


### Cleaning needs

#### Data cleaning: Fixing null values

Based on the above cell we find the following columns have null values:
| Column Name | Null Count |
|--------|-----------|
| personality_type | 3 |
| jury_status | 405 |
| original_tribe | 2 |
| swapped_tribe | 284 |
| swapped_tribe2 | 683 |
| merged_tribe | 300 |


### Methods

The following code cells implement the castaways cleaning steps on `clean_castaways_df`.


In [ ]:
#first we create a copy of the dataframe for editing
clean_castaways_df = castaways_df.copy()

#### personality_type
First we will fix the missing personality types. Since we are only missing three, we could just drop these rows, but we may want other information on these players, so we will fill them with 'UNKNOWN' instead.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['personality_type'].info())

In [ ]:
print(clean_castaways_df['personality_type'].unique())
clean_castaways_df['personality_type'] = clean_castaways_df['personality_type'].fillna('UNKNOWN')

In [ ]:
print("After: \n")
print(clean_castaways_df['personality_type'].info())

#### jury_status

In [ ]:
# See all unique values in the column
print(f"Before: \n")
print(clean_castaways_df['jury_status'].info())

In [ ]:
print(castaways_df['jury_status'].unique())

After viewing this we can understand that this column tells us what member of the jury a castaway is. The players eliminated earlier in the season don't make it on the jury, which explains why there are so many null values. After some consideration we will replace all null values with "non jury member" to follow the naming convention. 

In [ ]:
clean_castaways_df['jury_status'] = clean_castaways_df['jury_status'].fillna('non jury member')

In [ ]:
print("After: \n")
print(clean_castaways_df['jury_status'].info())

#### original_tribe
Since there are only two null values in this column, we will investigate which two castaways have the null values to determine if we should drop the rows or replace the null with a value.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['original_tribe'].info())

In [ ]:
#output the two null rows
print(castaways_df[castaways_df['original_tribe'].isnull()])

After some googling I found this explanation from a Reddit post: "For those of you who don’t know, in the very first episode of Palau (Season 10), Jonathan Libby and Wanda Shirk were eliminated before tribes were even officially formed." Based on this we can conclude that we can drop these rows from the dataframe since they are inconsequential to understanding themes in the show. 

In [ ]:
clean_castaways_df = clean_castaways_df.dropna(subset=['original_tribe'])


In [ ]:
print("After: \n")
clean_castaways_df['original_tribe'].info()

#### swapped_tribe, swapped_tribe2
We will handle swapped_tribe and swapped_tribe2 together since they contain the same type of information, but some castaways swap tribes once, twice, or never.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['swapped_tribe'].info())
print(clean_castaways_df['swapped_tribe2'].info())

In [ ]:
print(clean_castaways_df['swapped_tribe'].unique())
print(clean_castaways_df['swapped_tribe2'].unique())

From this output and some general knowledge, we can understand that these columns tell us what tribe a castaway switched to. However, not all players switch tribes, and especially most players don't switch tribes twice, but they are still important to the story the data is telling us. We will fill these with "Not Applicable".

In [ ]:
swap_columns = ['swapped_tribe', 'swapped_tribe2']
clean_castaways_df[swap_columns] = clean_castaways_df[swap_columns].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df[swap_columns].info()

#### merged_tribe

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['merged_tribe'].info())

In [ ]:
print(clean_castaways_df['merged_tribe'].unique())
print(f"Number of merged tribe names: {clean_castaways_df['merged_tribe'].nunique()}")

After viewing this list of the merged tribe names we now understand that this column represents the name of the merged tribe a castaway was in. Part way through the season all of the tribes get merged into one tribe, and they create a new name for the tribe. Since around half of the players each season get eliminated before the merge, said players have null values. We will fill this with 'Not Applicable' since we still want the data on the players who don't make it to the merge.

In [ ]:
clean_castaways_df['merged_tribe'] = clean_castaways_df['merged_tribe'].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df['merged_tribe'].info()

### Results


#### The data for the Castaways dataframe is now cleaned.

In [ ]:
clean_castaways_df.info()

## Table: Viewers


### Initial CSV exploration

#### Relevance:
- Useful for gauging interest in Survivor.
- May reveal relationships between public interest and contestant performance.

#### Size:
- 596 rows
- 9 columns
- 42 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **episode_number_overall**: The episode number across all seasons
- **episode**: The episode number within this season
- **title**: The title of the episode
- **episode_date**: The date the episode aired
- **viewers**: The number of viewers, in millions
- **rating_18_49**: Percentage of TV households in the 18-49 demographic that watched Survivor
- **share_18_49**: Among 18-49 viewers watching TV during the time slot, the percentage who watched Survivor

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
viewers_df.info()
viewers_df.head()

### Cleaning needs


First we make a copy of the dataframe to clean.

#### episode_number_overall
The overall episode numbers are stored as floats, but there is never an episode with a fraction, so we will convert this coumn to ints. First we need to investigate the single null value.

We can see that episode 2 of this season is 501, and after a google check as well, season 34 episode 1 of survivor is indeed the 500th episode, so we will manually change it, and then convert all of the values in this column to integers since an episode number won't be anything but a whole number. 

#### episode_date
episode_date is currently stored as an object, but we can convert it to datetime to be more accurate to what is is representing.

#### viewers, rating, share

Based on these observations it seems like there is just missing data for viewers, ratings, and the shares. We will take the average of each of these categories to fill in the data. This may skew things in a graph or other ways which we saw in class, but it is the best option to keep all of the episode details. 

The rating and share columns are confusing and lengthy with the addition of the 18_49 at the end, especially because there is no other age range as a different dataset, so there's no need for that distinction. We will change the column names to rating and share for clarity and ease.

### Methods


In [ ]:
clean_viewers_df = viewers_df.copy()

In [ ]:
null_idx = clean_viewers_df[clean_viewers_df['episode_number_overall'].isnull()].index[0]
print(clean_viewers_df.loc[null_idx - 1 : null_idx + 1])

In [ ]:
clean_viewers_df.loc[84, 'episode_number_overall'] = 500
clean_viewers_df['episode_number_overall'] = clean_viewers_df['episode_number_overall'].astype('int64')

In [ ]:
clean_viewers_df['episode_date'] = pd.to_datetime(clean_viewers_df['episode_date'])

In [ ]:
print("Before: \n")
print(clean_viewers_df['viewers'].info())
print("\n")
print(clean_viewers_df['rating_18_49'].info())
print("\n")
print(clean_viewers_df['share_18_49'].info())


In [ ]:

print(viewers_df[viewers_df['viewers'].isnull()].head())

In [ ]:
median_viewers = clean_viewers_df['viewers'].median()
median_rating = clean_viewers_df['rating_18_49'].median()
median_share = clean_viewers_df['share_18_49'].median()

clean_viewers_df['viewers'] = clean_viewers_df['viewers'].fillna(median_viewers)
clean_viewers_df['rating_18_49'] = clean_viewers_df['rating_18_49'].fillna(median_rating)
clean_viewers_df['share_18_49'] = clean_viewers_df['share_18_49'].fillna(median_share)

In [ ]:
clean_viewers_df = clean_viewers_df.rename(columns={
    'rating_18_49': 'rating',
    'share_18_49': 'share'
})

### Results


In [ ]:
print("After: \n")
print(clean_viewers_df['episode_number_overall'].info())

In [ ]:
print("After: \n")
print(clean_viewers_df['episode_date'].info())

In [ ]:
print("After: \n")
print(clean_viewers_df['viewers'].info())
print("\n")
print(clean_viewers_df['rating'].info())
print("\n")
print(clean_viewers_df['share'].info())

#### Viewers dataframe is now cleaned:

In [ ]:
print(clean_viewers_df.info())

## Table: Jury votes


### Initial CSV exploration

#### Relevance:
- Captures how jury members voted at Final Tribal Council.
- May reveal how personalities affect outcomes in the end.

#### Size:
- 909 rows
- 5 columns
- 35.6 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **castaway**: The juror casting votes
- **finalist**: A finalist for the season who can receive jury votes
- **vote**: Whether the juror voted for this finalist (1 = yes, 0 = no)

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
jury_votes_df.info()
jury_votes_df.head()

### Cleaning needs

No cleaning steps for this dataframe yet. There are no nulls, appropriate column names, and column types.


## Sources Used

- **Tool:** Cursor
- **How used:** Structure, cell order, format, PEP 8, grammatical errors, syntax errors, semantic errors
- **Scope:** Refinement, Debugging, Review

# Statistics

Now we focus on some statistics to determin what it takes to outwit, outplay, and outlast!

The following statistics will calculated and interpreted:
- Mean
- Median
- Standard Deviation
- Correlations


# Defining Winners & Losers

The following data are derived for a clean analysis phase

Identity
- `season` (int)
- `castaway` (short name)
- `full_name` (labels and tie-breaks)

Personality
- `personality_types` (16 MBTI codes & UNKNOWN)
  - may split into 4 seperat codes for E/I, S/N, T/F, J/P

Outcome for __'Outwit, Outplay, Outlast'__
- `order` (player longveity)
- `day` (days survived)
- `total_votes_received` (social risks)
- `
- `immunity_wins` (game perfomance, derived from `challenges_cleaned_df`)
- `reward_wins` (game perfomance, derived from `challenges_cleaned_df`)

Outcome
- `is_winner` (bool, derived from `castaway` == `winner`)

Time Axis for __'Evolution'__
- `season_year` (derived form `premiered`)

# Aggregate Castaway Dataframe

Create outplay_df to measure player's abbility to outplay others

From challenges_cleaned_df:
1. Group by `season`, `winners`, `challenge_type`, and count rows.
2. Pivot `challenge_type` to columns named `immunity_challange_wins` and `reward_challenge_wins`.
3. Reset the index and rename `winners` to `castaway`.

**Result**: a small lookup table with season, castaway, immunity wins, and reward wins.

In [ ]:
# Group by "season", "winners", "challenge_type" then count rows
outplay_df = challenges_cleaned_df.groupby(['season', 'winners', 'challenge_type']).size().reset_index(name='count')

# Pivot challenge_type to columns named immunity and reward
outplay_df = outplay_df.pivot(index=['season', 'winners'], columns='challenge_type', values='count').fillna(0).astype(int)

# Reset the index and rename winners to castaway
outplay_df = outplay_df.reset_index().rename(columns={'winners': 'castaway', 'immunity': 'immunity_challenge_wins', 'reward': 'reward_challenge_wins'})

# Remove challenge_type row label
outplay_df.columns.name = None

# Show Transformed Data
outplay_df.info()
outplay_df.head()


# Filter the Summary Dataframe

Create identity_df to for winner identification over time

From clean_summary_df:
1. Copy only `season`, `winner`, `premiered`.
2. Derive `season_year` from `premiered.dt.year` and drop `premiered`.

Result: identity_df with `season`, `winner`, `season_year`.

In [ ]:
# Copy only "season", "winner", "premiered"
identity_df = clean_summary_df[["season", "winner", "premiered"]].copy()
# Derive season_year from premiered.dt.year and drop premiered.
identity_df["season_year"] = identity_df["premiered"].dt.year
identity_df = identity_df.drop(columns=["premiered"])

# Show Transformed Data
identity_df.info()
identity_df.head()

# Filter Castaway Rows 

Create a traits_df for outlast, outwit, and personality analysis

From clean_castaways_df:
1. keep only: `season`, `castaway`, `full_name`, `personality_type`, `order`, `day`, `total_votes_received`, `immunity_idols_won`.
2. Rename `immunity_idols_won` → `immunity_idols_obtained`.

In [ ]:
# Filter the Castaway Dataframe
traits_df = clean_castaways_df[["season", "castaway", "full_name", "personality_type", "order", "day", "total_votes_received", "immunity_idols_won"]].copy()
traits_df.rename(columns={"immunity_idols_won": "immunity_idols_obtained"}, inplace=True)

# Show Transformed Data
traits_df.info()
traits_df.head()

# Merge New Dataframes for Analysis

1. Left-merge identity_df on season.
2. Left-merge the outplay_df on `season`, `castaway`.
3. fillna(0) on immunity_challenge_wins and reward_challenge_wins, cast to int for no win scenarios
4. Normalize `castaway` and `winner`
5. Create is_winner = normalized_castaway == normalized_winner.
6. Drop winner after the boolean is set.

In [ ]:
# left-merge identity_df on season
analysis_df = pd.merge(traits_df, identity_df, on='season', how='left')

# left-merge outplay_df on `season`, `castaway`
analysis_df = pd.merge(analysis_df, outplay_df, on=['season', 'castaway'], how='left')

# Fill NA values with 0 for immunity_challenge_wins and reward_challenge_wins
analysis_df[['immunity_challenge_wins', 'reward_challenge_wins']] = analysis_df[['immunity_challenge_wins', 'reward_challenge_wins']].fillna(0).astype(int)

# Normalize `castaway` and `winner`
analysis_df['castaway'] = analysis_df['castaway'].str.strip().str.casefold()
analysis_df['winner'] = analysis_df['winner'].str.strip().str.casefold()

# Create is_winner = normalized_castaway == normalized_winner.
analysis_df['is_winner'] = analysis_df['castaway'] == analysis_df['winner']

# Drop winner and castaway short names after the boolean is set.
analysis_df = analysis_df.drop(columns=['winner', 'castaway'])

# Re-order columns to something more intuitive
analysis_df = analysis_df[['season', 'full_name', 'personality_type', 'order', 'day', 'total_votes_received', 'immunity_idols_obtained', 'immunity_challenge_wins', 'reward_challenge_wins', 'is_winner', 'season_year']]

### Re-entry Season Mechanic

During season 38, one player was removed, re-entered and then won. The following cell attempts to remove duplicate winner entries, for all such cases.

In [ ]:
# Select duplicate winners
winner_mask = analysis_df["is_winner"]
dup_season_mask = analysis_df.groupby("season")["is_winner"].transform("sum") > 1
dup_mask = winner_mask & dup_season_mask

# Select index of duplicate winners
dup_index = analysis_df.loc[dup_mask].index

# Select index ofthe highest order winner
keep_index = analysis_df.loc[dup_mask].groupby(["season", "full_name"])["order"].idxmax()

# Drop duplicate winner rows that are not the highest order winner
drop_index = dup_index.difference(keep_index.values)
analysis_df = analysis_df.drop(drop_index)

# Show final anlaysis dataframe
analysis_df.info()
analysis_df.head()

# Initial Statistics
- Determine the  global mean, median, and standard deviations for key metrics
- Compute these again for winners only
- Compute them again for era
- Focused correlations for gameplay, logevity, and is_winner

## Global Statiscs - Everyone vs. Winners

In [ ]:
# Key columns for statistics
stats_cols = ["order", "day", "total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins"]

# Global Statistics for Analysis_df
print("Global Statistics for Analysis_df")
display(HTML(analysis_df[stats_cols].describe().style.format(precision=2).to_html()))

# Winners Only Statistics
print("\nWinners Only Statistics")
winner_df = analysis_df.loc[analysis_df["is_winner"], stats_cols]
display(HTML(winner_df.describe().style.format(precision=2).to_html()))

### Statistics Initial Thoughts

Everyone
- Order is essentially uniform
- Day clusters at the beginning and in finals ranges
- Votes received is skewed to the right, so a few people must collect many votes
- Idols obtained has an inflation of zeros, due to unavailbility in early seasons

Winners
- Go twice as far (day and order)
- Recieve about half the number of votes
- Roughly double the idols and challenge wins
- Winning requires surviving, so any metric that grows with longevity should be normalized


## Correlations

In [ ]:
# Correlation columns
stats_cols = ["order", "day", "total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins", "is_winner"]
print("Correlation Matrix for Everyone")
display(HTML(analysis_df[stats_cols].corr().style.format(precision=2).to_html()))

### Correlation Thoughts
Order and Day 
- Are essentially redundant with a 0.96 for showing longevity

Votes received 
- Is uncorrelated with all almost all statistics
- Has a modest negative correlation with winners
- Winners most likely avoid getting as many votes

Idols Obtained 
- Is the strongest indicator 
- Moderate correlation to longevity (order/days) 
- Modestly with winners 
- None with votes received
- They may matter more for longevity than for social risks.

Immunity and Reward Wins
- Track strongly with day, most likely due to tenure
- Only modestly for winners
- They appear to keep you in the game longer, but don't necessarily lead to winning.

Winners
- Modestly recieve less votes and more challenge wins
- Moderately receive idols

# Visual 1: Winners vs. Non-Winners across Eras
- How do winners and non-winners compare on average idols, immunity wins, and challenge wins across different eras?

## Strategy
1. Drop seasons with re-entry
2. Create season bins for each era (every 5 years)
3. Group by era and winner
4. Show key metrics

In [ ]:
outplay_analysis_df = analysis_df.copy()

#drop seasons with re-entry abilities, skews the data
outplay_analysis_df =outplay_analysis_df[~outplay_analysis_df["season"].isin([22, 23, 27, 38, 40])]

#sort from low to high seasons
outplay_analysis_df = outplay_analysis_df.sort_values(by="season")

#create bins to group seasons into eras of the game
season_bins = [1999, 2004, 2009, 2014, 2020]
season_labels = ["2000-2004", "2005-2009", "2010-2014", "2015-2020"]

#add new column for eras
outplay_analysis_df["era"] = pd.cut(
    outplay_analysis_df["season_year"],
    bins = season_bins,
    labels = season_labels
)

era_means = outplay_analysis_df.groupby(["era", "is_winner"])[["immunity_challenge_wins", "reward_challenge_wins", "immunity_idols_obtained"]].mean()
era_means = era_means.reset_index()

#manually edit immunity_idols for 2000-2004, there were no idols introduced until 2005
era_means.loc[era_means["era"] == "2000-2004", "immunity_idols_obtained"] = 0

#visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.barplot(
    data=era_means,
    x="era",
    y="immunity_challenge_wins",
    hue="is_winner",
    ax=axes[0]
)

sns.barplot(
    data=era_means,
    x="era",
    y="reward_challenge_wins",
    hue="is_winner",
    ax=axes[1]
)

sns.barplot(
    data=era_means,
    x="era",
    y="immunity_idols_obtained",
    hue="is_winner",
    ax=axes[2]
)


#formatting
axes[0].set_title("Immunity Challenge Wins by Era")
axes[0].set_xlabel("Era")
axes[0].set_ylabel("Mean Wins")

axes[1].set_title("Reward Challenge Wins by Era")
axes[1].set_xlabel("Era")
axes[1].set_ylabel("Mean Wins")

axes[2].set_title("Immunity Idols Obtained by Era")
axes[2].set_xlabel("Era")
axes[2].set_ylabel("Mean Idols")

#normalize challenge and reward wins to the same y range
axes[0].set_ylim(0, 8)
axes[1].set_ylim(0, 8)
axes[2].set_ylim(0, 2.5)

#remove redundant labels
axes[0].get_legend().remove()
axes[1].get_legend().remove()
axes[2].legend(title="Winner", labels=["Non-Winner", "Winner"], handles=axes[2].legend_.legend_handles)

#final formatting
fig.suptitle("Winner vs Non-Winner Performance Across Eras", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


*Footnote: Immunity idols were not introduced into survivor unti Season 11 in 2005, but the dataframe has values for idols in the 2000-2004 era. This may be representing other advantages, but for our purposes I manually zero-ed out this era to avoid confusion.*

## Observations

The purpose of this visualization was to compare what winners versus losers of Survivor looked like over the different eras of survivor. We were expecting to see challenge and reward wins to decrease over the eras as strategy became more complicated and intense, and players seemed more focused on eliminating strong competitors earlier. 

Surprisingly, based on the graphs we can see that winners have dominated immunity challenges around 2x more, and reward challenges around 2.3x more. This directly challenges the expectation that "challenge-beasts" don't win Survivor. The only exception is the pre-idol era (2000-2004), where the gap between losers and winners is smaller. 

When idols were introduced, it seems like winners had the protection of idols to play more dominantly in challenges without the fear of being voted off. Additonally, players who had both idols and were winning immunity got harder and harder to vote out. This not only proves the outplay component, but also the outlast and outwit. Based on the graph, winners find around 3x more idols than losers. To find idols you have to outwit opponents by understanding where clues may be hidden, and then figuring out how to go look for an idol without your teammates catching on. Additionally, the added protection of an idol allows a player to outlast even when being voted off. 

It should be acknowleged that the winner sample is much smaller per era (8-10 winners versus 125-171 losers), which could skew data. Within the losers sample are contestants who only lasted a few days versus contestants who lasted until the end. 

# Visual 2: Personality Type (MBTI) & Winners
- How dow the MBTI codes correlate with winners?

# Strategy
1. Drop unkowns
2. Split personality MBTI code into 4 boolean letter codes
3. Normalize columns that naturally grow with time

In [ ]:
# Create a copy for the MBTI correlation heatmap
mbti_heatmap_df = analysis_df.copy()

# Rename idol column if the notebook used a different final name
if "immunity_idols_obtained" not in mbti_heatmap_df.columns:
    if "immunity_idols_found" in mbti_heatmap_df.columns:
        mbti_heatmap_df = mbti_heatmap_df.rename(
            columns={"immunity_idols_found": "immunity_idols_obtained"}
        )

# Keep only rows with a real MBTI type
mbti_heatmap_df = mbti_heatmap_df[
    mbti_heatmap_df["personality_type"] != "UNKNOWN"
].copy()

# Create MBTI letter flags
# These use one side of each MBTI pair to avoid duplicate opposite columns
mbti_heatmap_df["is_extrovert"] = (
    mbti_heatmap_df["personality_type"].str[0] == "E"
).astype(int)

mbti_heatmap_df["is_intuitive"] = (
    mbti_heatmap_df["personality_type"].str[1] == "N"
).astype(int)

mbti_heatmap_df["is_thinking"] = (
    mbti_heatmap_df["personality_type"].str[2] == "T"
).astype(int)

mbti_heatmap_df["is_judging"] = (
    mbti_heatmap_df["personality_type"].str[3] == "J"
).astype(int)

# Convert winner label to 0/1 for correlation
mbti_heatmap_df["winner_flag"] = mbti_heatmap_df["is_winner"].astype(int)

# Normalize metrics that naturally grow with time in the game
mbti_heatmap_df["survival_pct"] = (
    mbti_heatmap_df["order"] /
    mbti_heatmap_df.groupby("season")["order"].transform("max")
)

mbti_heatmap_df["votes_per_day"] = (
    mbti_heatmap_df["total_votes_received"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["idols_per_day"] = (
    mbti_heatmap_df["immunity_idols_obtained"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["immunity_wins_per_day"] = (
    mbti_heatmap_df["immunity_challenge_wins"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

mbti_heatmap_df["reward_wins_per_day"] = (
    mbti_heatmap_df["reward_challenge_wins"] /
    mbti_heatmap_df["day"].clip(lower=1)
)

# Select MBTI dimensions and outcome/gameplay metrics
mbti_columns = [
    "is_extrovert",
    "is_intuitive",
    "is_thinking",
    "is_judging"
]

metric_columns = [
    "winner_flag",
    "survival_pct",
    "votes_per_day",
    "idols_per_day",
    "immunity_wins_per_day",
    "reward_wins_per_day"
]

# Create a focused correlation table
mbti_correlation_matrix = mbti_heatmap_df[
    mbti_columns + metric_columns
].corr()

mbti_correlation_focus = mbti_correlation_matrix.loc[
    mbti_columns,
    metric_columns
].round(2)

mbti_correlation_focus

In [ ]:
mbti_correlation_plot = mbti_correlation_focus.rename(
    index={
        "is_extrovert": f"Extrovert (+) vs.\nIntrovert (-)",
        "is_intuitive": f"Intuitive (+) vs.\nSensing (-)",
        "is_thinking": f"Thinking (+) vs.\nFeeling (-)",
        "is_judging": f"Judging (+) vs.\nPerceiving (-)"
    },
    columns={
        "winner_flag": "Winner",
        "survival_pct": "Survival %",
        "votes_per_day": "Votes per day",
        "idols_per_day": "Idols per day",
        "immunity_wins_per_day": "Immunity wins per day",
        "reward_wins_per_day": "Reward wins per day"
    }
)

# Draw Heat Map
fig, ax = plt.subplots(figsize=(12, 5.5))

sns.heatmap(
    data=mbti_correlation_plot,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Curtailed Correlation"}
)

ax.set_title(
    "MBTI Letter Correlation With Survivor Performance Metrics",
    fontsize=16,
    pad=14
)

ax.set_xlabel("Performance Metrics", labelpad=12)
ax.set_ylabel("MBTI Letter Flags", labelpad=18)

ax.tick_params(axis="x", labelrotation=35, labelsize=10)
ax.tick_params(axis="y", labelrotation=0, labelsize=11, pad=8)

plt.tight_layout()
plt.show()

### MBTI Correlation Heatmap Observation

This heatmap compares MBTI letter dimensions against winner status and normalized Survivor performance metrics. I split each personality type into broader letter flags because the full four-letter MBTI groups are small, while the Extrovert/Introvert (E/I), Sensing/Intuitive (S/N), Thinking/Feeling (T/F), and Judging/Perceiving (J/P) dimensions give a cleaner view of the data.

The correlations are mostly weak, which means MBTI alone does not strongly explain who wins or performs well. Still, a few small patterns are worth noticing. Thinking types show the strongest positive relationship with winner status and idols per day, while judging types trend slightly lower across challenge-win metrics. These are not definitive conclusions, but they give us a useful starting point for asking better questions.

I also normalized votes, idols, and challenge wins by days survived because raw totals naturally favor players who lasted longer. That keeps the visual more objectively correct. Instead of treating personality as the be all and end all, this heatmap treats it as one measure among many being that of a small signal inside the much bigger social game.

In [ ]:
print("MBTI heatmap rows:", len(mbti_heatmap_df))
print("\nColumns used:")
print(mbti_columns + metric_columns)

print("\nMissing values:")
print(mbti_heatmap_df[mbti_columns + metric_columns].isna().sum())

# Visual 3: Winners & Votes
- How are winners receiving votes compared to non-winners?

## Strategy
1. Normalize total votes received according to time
2. Use a violin chart to illustrate

In [ ]:
#Create a copy of the analysis dataframe
votes_outcome_df = analysis_df.copy()

# Normalize votes by days survived
votes_outcome_df["votes_per_day"] = (
    votes_outcome_df["total_votes_received"] /
    votes_outcome_df["day"].clip(lower=1)
)

vote_cap = votes_outcome_df["votes_per_day"].quantile(0.99)
votes_outcome_df["vote_plot"] = votes_outcome_df["votes_per_day"].clip(upper=vote_cap)

# Violin plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    x="is_winner",
    y="vote_plot",
    ax=ax,
    inner="box",
    data=votes_outcome_df,
)

# Formatting
ax.set_title("Votes per Day Survived by Winner Status")
ax.set_xlabel("Winner")
ax.set_ylabel("Votes per Day")
fig.text(0.5, -0.1, "Note: Capping at 99th percentile to reduce outlier influence", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Observation

### Notes
From the early global statistics, the non-winners have a approximately double the median and mean votes received than winners. The violin chart above accounts for votes received per day, to focus in on the rate of accumulation rather than total votes received. 

Limitations: The non-winner group mixes in people who were voted off early, so the votes recieved per day has big swings, and a few people collect many of votes. The length of its violin partly reflects game length and how often someone is targeted.

### Thoughts
It appears that the majority of winners receive a very low number of votes per day, whereas non-winners have a wider distribution of votes at a higher rate per day then winners.

# Phase 3 Questions Answering

The major areas explored are:
1. Effects of Survival Length on Winner and Non-Winner Metric Comparisons
2. Idols vs. Winners and How Their Interactions Have Changed Over Time
3. Deep MBTI Analysis of Representation and Tenure Interactions

## Question on Effects of Survival Length on Metrics
After normalizing accumulating metrics by days survived and binning each season into early, middle, and late phases:

How do winner vs. non-winner metric gaps change when comparing the following cohorts?
- Players who left in the same phase (boot cohorts)?
- Players who survived the same phase (survival cohorts)?

Does this pattern differ by era?

### Motivation
It seems that time affects the comparisons between competitors more than any other feature. In order to combat this, the data should be normalized against time. Also, seasons should be binnned per era, and individual seasons should be binned by depth. This helps deal with changes in the game over eras, and accumulation of data during the length of seasons. For example, players in the first few seasons did not have access to idols, which is a key feature in later seasons. Also during the course of a season, players who are voted off early had no chance to accumulate any data on challenge wins.

### Step 1: Normalize Columns by Day and Order

- Normalizing by `day` will be used on vote, idols, and challenge wins.
- Normalizing by `order` will be used to determine survival depth and quartiles.


In [ ]:
# Create a copy of the analysis dataframe
depth_df = analysis_df.copy()

# Normalize columns that accumulate over time
num_days = depth_df["day"].clip(lower=1) # clip to avoid division by 0

depth_df["votes_per_day"] = depth_df["total_votes_received"] / num_days
depth_df["idols_per_day"] = depth_df["immunity_idols_obtained"] / num_days
depth_df["immunity_wins_per_day"] = depth_df["immunity_challenge_wins"] / num_days
depth_df["reward_wins_per_day"] = depth_df["reward_challenge_wins"] / num_days

# Calculate survival depth as a percentage of remaining players
# i.e. what percentage of players did they outlast?
# Higher order number means they lasted longer
max_players = depth_df.groupby("season")["order"].transform("max")
depth_df["survival_pct"] = depth_df["order"] / max_players

# Drop unneeded columns
depth_df = depth_df.drop(columns=["total_votes_received", "immunity_idols_obtained", "immunity_challenge_wins", "reward_challenge_wins"])

### Step 2: Bin seasons by Era

- Ideally this will be used for filtering out changes across seasons
  - Particularly the Pre-2005 era, which did not have access to idols
- Note: this transfomation matches the bin sizes used earlier

In [ ]:
# Create bins for each era (every 5 seasons)
era_bins = [1999, 2004, 2009, 2014, 2020]
era_labels = ["2000-2004", "2005-2009", "2010-2014", "2015-2020"]

# Bin the data
depth_df["era"] = pd.cut(depth_df["season_year"], bins=era_bins, labels=era_labels)

### Step 3: Bin Depth within each Season
Motivation
- Early contestants have little to no collection of data when compared to contestants that lasted longer.
- Binning by depth allows for comparisons between players who survived a similar amount of time
- This allows for comparisons between players who were booted early, or for more specific analysis on players who made it almost to the end.

Strategy
- Bin on tertiles for an early, middle, late game comparison
- Create a column that identifies when a conestant was booted
- Create columns that identify the phases the contestants survived
- This should have larger groups and is convenient to reason about

In [ ]:
# Phases of the game
PHASES = ["early", "middle", "late"]
CUTS = [0, 1/3, 2/3]
BINS = CUTS + [1.0] # for 3 bins, need 4 edges

# Create a column that shows the phase the player was booted
depth_df["boot_phase"] = pd.cut(
    depth_df["survival_pct"],
    bins=BINS,
    labels=PHASES,
    include_lowest=True, # include the lowest value
    right=True, # right edge is inclusive
)

# Create columns that shows if a player was present at each phase
for phase, cut in zip(PHASES, CUTS):
    depth_df[f"{phase}"] = depth_df["survival_pct"] >= cut


### Step 4: Boot Phase Comparison
Compares cohorts of players that were booted (voted off) at the same time, across eras and whether they won or not.

Note:
- Winners only show in the final boot cohort, as they lasted the entirety of the show and were technically never booted.
- The median is used here as its slighty more resistant to outliers (i.e. vote spikes)

In [ ]:
# Columns to use for phase comparison
RATE_COLS = ["votes_per_day", "idols_per_day", "immunity_wins_per_day", "reward_wins_per_day"]

# Create Median Summary Group by era, depth tertile, and winner status
# Only observe combinations that have occurred (observed=True)
boot_summary = (
    depth_df.groupby(["era", "boot_phase", "is_winner"], observed=True)[RATE_COLS]
    .median()
    .round(4)
)

### Step 5: Survival Phase Comparison
Compares players that were still present on the show for each phase, and whether they were eventually winners or not.

Note:
- Winners are visible within each cohort, as they are present for the entirety of the show.
- The median is used here as its slighty more resistant to outliers (same as above)

In [ ]:
# Create a df to represent the survival cohorts of players
survival_df = depth_df.copy()

# Create a long version of the dataframe for viewing
survival_df = survival_df.melt(
    id_vars=["era", "is_winner"] + RATE_COLS,
    value_vars=PHASES,
    var_name="phase",
    value_name="present"
)

# Make phase an ordered categorical for better visualization
survival_df["phase"] = pd.Categorical(survival_df["phase"], categories=PHASES, ordered=True)

# Keep only the rows where the player was present at the given phase
survival_df = survival_df[survival_df["present"]]

# Create Median Summary Group by era, winner status, and phase
survival_summary = (
    survival_df.groupby(["era", "phase", "is_winner"], observed=True)[RATE_COLS]
    .median()
    .round(4)
)

### Step 6: Visualizations

The following visuals will do the following:
- Show the differences in metrics between the players that were booted in the late game with winners. 
- Show the differences in metrics between players who survived each phase with winners and non-winners.

**Bar Chart Late Phase Comparison**

This shows how the median metrics compare between players who made it to the final phase of the game show only, in order to explore how good competitors compare with the people who won.

**Heatmap Survival Phase Comparison**

This shows how the median metrics compare between winners and non-winners change throughout each phase and each era. It illustrates the difference in the median metrics, it is not a correlation map.

In [ ]:
# Define readable labels for the metrics
AXIS_LABELS = {
    "votes_per_day": "Votes / day",
    "idols_per_day": "Idols / day",
    "immunity_wins_per_day": "Immunity Wins / day",
    "reward_wins_per_day": "Reward Wins / day",
}


def shorten_era(era):
    """Shortens era labels for axis labels, by removing the first two characters."""
    a, b = str(era).split("-") # split on the dash
    return f"{a[2:]}-{b[2:]}" # slice the first two characters off each


def metric_difference(survival_df, metric, eras):
    """
    Computes difference between winner and non-winner medians for a given metric.
    Rows should be eras, columns should be game phases (early / middle / late).
    """
    # Reset index to make the dataframe flat
    df = survival_df.reset_index() 
    # This will create two dataframes, one for winners and one for non-winners.
    # Pivot the dataframe to have eras as rows, phases as columns, and values as the metric.
    win = df[df["is_winner"]].pivot(index="era", columns="phase", values=metric)
    lose = df[~df["is_winner"]].pivot(index="era", columns="phase", values=metric)
    # Compute the difference and reindex to match the era order.
    med_diff = (win - lose).reindex(index=eras, columns=PHASES)
    # Format the era labels
    med_diff.index = [shorten_era(e) for e in eras]
    return med_diff


def plot_boot_cohort_bars(boot_df):
    """
    Boot Cohort Visualization (Late Boots Only)
    - Shows the median rate for each metric by era, separated by winner status in a bar chart.
    """
    metrics = list(RATE_COLS)

    # Create a dataframe for the boot cohort and shorten the era labels
    late_df = boot_df.reset_index().query("boot_phase == 'late'").copy()
    late_df["era_short"] = late_df["era"].astype(str).map(shorten_era)

    # Create a figure with one row and one column for each metric
    fig, axes = plt.subplots(
        1, len(metrics),
        figsize=(5 * len(metrics), 4.5),
        squeeze=False,
    )

    # Plot each metric
    for col, metric in enumerate(metrics):
        label = AXIS_LABELS[metric]
        ax = axes[0, col]
        sns.barplot(
            data=late_df, x="era_short", y=metric,
            hue="is_winner", hue_order=[False, True], ax=ax,
        )
        ax.set_title(label, fontweight="bold")
        ax.set(xlabel="Era", ylabel=label)
        ax.grid(axis="y", alpha=0.3)
        ax.get_legend().remove() # Remove the legend for the bar chart

    # Create a legend for all the bar charts
    handles, _ = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, ["Non-winner", "Winner"], title="Outcome",
        loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2,
    )
    # Set the title of the figure
    fig.suptitle(
        "Winner vs. Runner-Up Median Rates by Era",
        fontsize=14, fontweight="bold", y=1.05,
    )
    # Adjust the layout of the figure
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    # Show the figure
    plt.show()

def plot_survival_cohort_heatmaps(survival_df, eras):
    """
    Survival Cohort Visualization (All Players)
    - Shows the winner vs non-winner median difference for each metric by era and phase in a heatmap.
    """
    metrics = list(RATE_COLS)

    # Create a figure with one row and one column for each metric
    fig, axes = plt.subplots(
        1, len(metrics),
        figsize=(5 * len(metrics), 5),
        squeeze=False,
    )

    # Plot each metric
    for col, metric in enumerate(metrics):
        label = AXIS_LABELS[metric]
        ax = axes[0, col]
        sns.heatmap(
            metric_difference(survival_df, metric, eras),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=0.5,
            cbar_kws={"label": "Winner - Non-winner (Δ)"},
            ax=ax,
        )
        ax.invert_yaxis()  # eras read early to late, bottom to top
        ax.set_title(label, fontweight="bold")
        ax.set(xlabel="Game Phase", ylabel="Era" if col == 0 else "")

    fig.suptitle(
        "Winner vs. Non-winner Median Difference by Era and Phase",
        fontsize=14, fontweight="bold", y=1.05,
    )
    fig.tight_layout()
    plt.show()

# Plot the visuals, and order the eras
eras = list(depth_df["era"].cat.categories)
plot_boot_cohort_bars(boot_summary)
plot_survival_cohort_heatmaps(survival_summary, eras)


### Conclusions

**Original Question:**
How do winner vs. non-winner metric differences change when comparing the following cohorts?
- Players who left in the same phase (boot cohorts)?
- Players who survived the same phase (survival cohorts)?

Does this pattern differ by era?

---

**Summary:** Vote differences were the most consistent signal, winners had lower votes/day in both views. Immunity and idol differences varied more by era. The survival cohort heatmaps made those era and phase patterns the easiest to read.

**Players Who Left in the Same Phase**

As a limitation, we could only meaningfully compare winners and non-winners in the late boot cohort, since winners always lasted until the end. However, in that group, winners had lower votes/day than late-boot non-winners in every era, even as both group medians crept up slightly over time. Immunity and idol rates were higher for winners mainly in 2010–2014, with flatter gaps in 2000–2004 and 2015–2020 (idol rates were particularly inconsistent). Reward win differences were smaller and less consistent, so they look less central to the winner profile in this view.

**Players Who Survived the Same Phase**

Across eras and phases, winners had lower median votes/day than non-winners still in the game at that phase. The gap was largest in early and middle phases and smallest in late phase. That fits fewer players remaining as the season progresses, and winners generally taking less vote heat than others, with the advantage narrows near the end.

Winners earned idols at a slightly higher rate than non-winners in most era phase cells, with almost no gap in the 2015–2020 late phase. The standout was 2010–2014, when idol rates were clearly higher for winners, suggesting idols mattered more for winning in that era than in others.

For immunity wins, there was a clear era shift: in 2000–2009, winner and non-winner medians were similar or slightly favored non-winners; from 2010 onward, winners were ahead, especially in 2010–2014. Within a season, immunity gaps did not change much from early to late phase.

Reward-win gaps were weaker and era-specific (for example, larger for winners in 2005–2009) rather than a steady mirror image of immunity. We would not read a clean “shift from rewards to immunities” from these medians alone, as early tribal wins also mix team and individual credit. Separating the individual metrics from team metrics is another limitation with this data set.

## Idols Vs. Winners - A Two Part Question

### Part 1 - The target question:    
*Does the relationship between idols obtained and votes received change across eras? (Early: negative or flat correlation. Modern: positive correlation)*    
### Part 2 — The survival question:  
*Among people who DO attract more votes, are winners better at surviving than non-winners, and does that differ by era?*


### Motivation
From a viewers perspective, in the earlier seasons when the immunity idol was first introduced, it acted as a security blanket, where people played around them or tried to form an alliance with them. In the modern era, the immunity idol becomes more of a target than security, with idol flushing and blindsides becoming the norm. In theory, the correlation between immunity idols obtained and total votes received should flip across eras. To dive deeper into the idols, are winners able to play idols differently? Do they accumulate more votes and neutralize them from an idol, or are they able to avoid votes even with the added threat of an idol? 

### Setting up dataframe to answer question

In [ ]:
question_df = analysis_df.copy()
question_df =question_df[~question_df["season"].isin([22, 23, 27, 38, 40])]

#sort from low to high seasons
question_df = question_df.sort_values(by="season")

#create bins to group seasons into eras of the game
season_bins = [1999, 2004, 2009, 2014, 2020]
season_labels = ["2000-2004", "2005-2009", "2010-2014", "2015-2020"]

#add new column for eras
question_df["era"] = pd.cut(
    question_df["season_year"],
    bins = season_bins,
    labels = season_labels
)

#filter out the 2000-2004 era since immunity idols weren't introduced until after this era
question_df = question_df[question_df["era"] != "2000-2004"]

question_df["era"] = question_df["era"].cat.remove_unused_categories()

### Answering Part 1: Finding the correlation between idols and votes per era

In [ ]:
era_correlations = question_df.groupby("era").apply(
    lambda x: x["immunity_idols_obtained"].corr(x["total_votes_received"])
)

print(era_correlations)

### Visualize Answer to Pt. 1

In [ ]:
sns.lmplot(
    data=question_df,
    x="immunity_idols_obtained",
    y="total_votes_received",
    col="era",
    height=4,
    aspect=0.8
)

### Findings from Part 1:
We can see from both the numerical output and graphical representation that there was in fact a negative correlation between votes and idols in the early seasons, but that as time has gone on the correlation has dropped to basically zero, meaning that the gameplay evolved to neutralize idols instead of viewing their holders as untouchable.

### Answering Part 2: Do winners use idols more successfully? 

In [ ]:
idol_holders_df = question_df[question_df["immunity_idols_obtained"] > 0]

winner_correlations = idol_holders_df.groupby(["era", "is_winner"])["total_votes_received"].agg('mean')

winner_correlations = winner_correlations.reset_index()

print(winner_correlations)

### Visualize Answer to Part 2

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

sns.barplot(
    data=winner_correlations,
    x="era",
    y="total_votes_received",
    hue="is_winner",
    ax=ax
)

ax.set_title("Mean Votes Received by Era for Idol Holders (Winners vs Non-Winners)", fontsize=14, fontweight="bold")
ax.set_xlabel("Era")
ax.set_ylabel("Mean Votes Received")

handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Non-Winner", "Winner"], title="Winner")

plt.tight_layout()
plt.show()

print(winner_correlations.head())
print(winner_correlations.columns.tolist())


### Findings from Part 2:
We can see from both the numerical and graphical outputs that between winners with idols and non-winners with idols, winners receieve significantly less votes across all eras. This implies that it's less about having an idol, and more about how a player uses it. 

### Findings

### Part 1 - The target question:    
#### *Does the relationship between idols obtained and votes received change across eras? (Early: negative or flat correlation. Modern: positive correlation)*       
We found that the security blanket effect that was hypothesized was correct. In the 2005-2009 era, having an idol had a negative correlation with votes (-0.28). Basically, having an idol meant you weren't getting voted for. By the 2015-2020 era, the correlation has basically dropped to 0 (-0.009). This can also be seen in the trend lines shifting from negative slop to almost 0 slope in my visualizations. From a viewer's perspective, in the earlier era players were more forthcoming about having an idol, and the mentality was that they couldn't really be voted for. But as the gameplay got more strategic and sneaky, having an idol didn't guarantee security anymore. 
### Part 2 — The survival question:  
#### *Among people who DO attract more votes, are winners better at surviving than non-winners, and does that differ by era?*      
In every era we can see that winners who have idols receieve fewer votes than non-winners who had idols. This gap has also widened over time, which you can see clearly in the visualization. Additionally, the non-winner idol holders get increasingly more votes over time. This implies that they are being targeted for having an idol. 
   ### Conclusion
Both of these findings contibute to a larger story about Survivor winners. Idols alone aren't protection, as the game has evolved to neutralize them easier. However, winners are able to leverage their idols as part of their broader social game, while keeping their perceieved threat level low. This could be due to a number of different factors, but as a viewer the biggest determining factor is choosing who to tell about having an idol. If a player puts their trust in the right people, they can use the idol as a team to advance all of their games, but if they tell the wrong people it can leave to a blindside that gets them voted off. We can conclude that idols can be a determining factor on winning versus losing, but only in combination with social positioning. 

## MBTI Question Answering

For Phase 3, we are narrowing our analysis to the MBTI patterns that came out of our Phase 2 heatmap work. The heatmap showed mostly weak correlations between MBTI letter flags and Survivor performance metrics, so our next step is not to claim that personality predicts winning. The better question is whether personality becomes more meaningful when we look at representation, rarity, era, and tenure.

These questions are intentionally very simple, but they still go beyond just looking at the data:

1. Are there MBTI types that are underrepresented, but win more often than others?
2. Are rare MBTI types different from common MBTI types in winner rate and survival depth?
3. Which eras do different MBTI types win more often in?
4. Do certain MBTI types last longer than others?

The goal is to treat MBTI as a thought experiment, not an end all be all. Survivor is still a social game built on timing, relationships, advantages, and perception.

In [ ]:
# Phase 3 setup for MBTI question answering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

# Work from the merged analysis dataframe
phase3_df = analysis_df.copy()


# Helper function to safely choose a column from possible names
def choose_column(df, possible_names, friendly_name):
    for name in possible_names:
        if name in df.columns:
            return name

    raise KeyError(
        f"Could not find a column for {friendly_name}. "
        f"Checked these names: {possible_names}"
    )


name_col = choose_column(
    phase3_df,
    ["full_name", "castaway", "castaway_name"],
    "contestant name"
)

mbti_col = choose_column(
    phase3_df,
    ["personality_type", "mbti", "MBTI"],
    "MBTI personality type"
)

winner_col = choose_column(
    phase3_df,
    ["is_winner", "winner"],
    "winner label"
)

season_col = choose_column(
    phase3_df,
    ["season"],
    "season"
)

season_year_col = choose_column(
    phase3_df,
    ["season_year", "year"],
    "season year"
)

order_col = choose_column(
    phase3_df,
    ["order", "placement"],
    "finish order"
)

day_col = choose_column(
    phase3_df,
    ["day", "days"],
    "days lasted"
)

votes_col = choose_column(
    phase3_df,
    ["total_votes_received", "votes_received"],
    "total votes received"
)

idols_col = choose_column(
    phase3_df,
    ["immunity_idols_found", "immunity_idols_obtained"],
    "immunity idols"
)

immunity_wins_col = choose_column(
    phase3_df,
    ["immunity_challenge_wins"],
    "immunity challenge wins"
)

reward_wins_col = choose_column(
    phase3_df,
    ["reward_challenge_wins"],
    "reward challenge wins"
)


# Standardize the columns used in this section
phase3_df["player_name"] = phase3_df[name_col]
phase3_df["personality_type_clean"] = (
    phase3_df[mbti_col]
    .astype(str)
    .str.upper()
    .str.strip()
)

phase3_df["is_winner_bool"] = phase3_df[winner_col].astype(bool)
phase3_df["season_num"] = phase3_df[season_col]
phase3_df["season_year_clean"] = phase3_df[season_year_col]
phase3_df["finish_order"] = phase3_df[order_col]
phase3_df["days_lasted"] = phase3_df[day_col]
phase3_df["votes_received"] = phase3_df[votes_col]
phase3_df["idols_found"] = phase3_df[idols_col]
phase3_df["immunity_wins"] = phase3_df[immunity_wins_col]
phase3_df["reward_wins"] = phase3_df[reward_wins_col]

# Keep only real four-letter MBTI types
valid_mbti_mask = phase3_df["personality_type_clean"].str.match(
    r"^[EI][SN][TF][JP]$",
    na=False
)

mbti_phase3_df = phase3_df.loc[valid_mbti_mask].copy()

# Normalize depth so seasons with different cast sizes can be compared
mbti_phase3_df["season_max_order"] = (
    mbti_phase3_df.groupby("season_num")["finish_order"].transform("max")
)

mbti_phase3_df["survival_pct"] = (
    mbti_phase3_df["finish_order"] /
    mbti_phase3_df["season_max_order"]
)

# Per-day metrics keep tenure from dominating every comparison
mbti_phase3_df["day_safe"] = mbti_phase3_df["days_lasted"].clip(lower=1)
mbti_phase3_df["votes_per_day"] = (
    mbti_phase3_df["votes_received"] / mbti_phase3_df["day_safe"]
)
mbti_phase3_df["idols_per_day"] = (
    mbti_phase3_df["idols_found"] / mbti_phase3_df["day_safe"]
)
mbti_phase3_df["immunity_wins_per_day"] = (
    mbti_phase3_df["immunity_wins"] / mbti_phase3_df["day_safe"]
)
mbti_phase3_df["reward_wins_per_day"] = (
    mbti_phase3_df["reward_wins"] / mbti_phase3_df["day_safe"]
)

# Era bins based on the project discussion
mbti_phase3_df["era"] = pd.cut(
    mbti_phase3_df["season_year_clean"],
    bins=[1999, 2004, 2009, 2014, 2020],
    labels=["2000-2004", "2005-2009", "2010-2014", "2015-2020"],
    include_lowest=True
)

# MBTI letter dimensions
mbti_phase3_df["E_or_I"] = mbti_phase3_df["personality_type_clean"].str[0]
mbti_phase3_df["S_or_N"] = mbti_phase3_df["personality_type_clean"].str[1]
mbti_phase3_df["T_or_F"] = mbti_phase3_df["personality_type_clean"].str[2]
mbti_phase3_df["J_or_P"] = mbti_phase3_df["personality_type_clean"].str[3]

print("Rows in analysis_df:", len(analysis_df))
print("Rows with valid MBTI:", len(mbti_phase3_df))
print("MBTI types found:", mbti_phase3_df["personality_type_clean"].nunique())

display(
    mbti_phase3_df[
        [
            "player_name",
            "personality_type_clean",
            "is_winner_bool",
            "season_num",
            "season_year_clean",
            "era",
            "finish_order",
            "days_lasted",
            "survival_pct"
        ]
    ].head()
)

## Question 1: Are there MBTI types that are underrepresented, but win more often than others?

Our Phase 2 heatmap suggested that MBTI does not have a strong direct correlation with winning. However, full MBTI types are small groups. A type can be rare in the cast, but still have a higher winner rate than expected. This question checks whether any underrepresented personality types stand out after accounting for how many players of each type appear in the dataset.

In [ ]:
# Question 1: Underrepresented MBTI types and winner rate

mbti_type_summary = (
    mbti_phase3_df
    .groupby("personality_type_clean", observed=True)
    .agg(
        players=("player_name", "count"),
        winners=("is_winner_bool", "sum"),
        median_survival_pct=("survival_pct", "median"),
        median_days_lasted=("days_lasted", "median"),
        median_votes_per_day=("votes_per_day", "median"),
        median_immunity_wins_per_day=("immunity_wins_per_day", "median")
    )
    .reset_index()
    .rename(columns={"personality_type_clean": "mbti_type"})
)

mbti_type_summary["winner_rate"] = (
    mbti_type_summary["winners"] / mbti_type_summary["players"]
)

mbti_type_summary["representation_share"] = (
    mbti_type_summary["players"] / mbti_type_summary["players"].sum()
)

# Below-median player count means underrepresented for this dataset
type_count_median = mbti_type_summary["players"].median()
overall_winner_rate = (
    mbti_type_summary["winners"].sum() / mbti_type_summary["players"].sum()
)

mbti_type_summary["representation_group"] = np.where(
    mbti_type_summary["players"] < type_count_median,
    "Underrepresented",
    "More common"
)

mbti_type_summary["above_overall_winner_rate"] = (
    mbti_type_summary["winner_rate"] > overall_winner_rate
)

underrepresented_winners = (
    mbti_type_summary[
        (mbti_type_summary["representation_group"] == "Underrepresented") &
        (mbti_type_summary["above_overall_winner_rate"])
    ]
    .sort_values(["winner_rate", "players"], ascending=[False, False])
)

display(
    mbti_type_summary
    .sort_values(["winner_rate", "players"], ascending=[False, False])
    .style.format(
        {
            "winner_rate": "{:.3f}",
            "representation_share": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "median_days_lasted": "{:.1f}",
            "median_votes_per_day": "{:.3f}",
            "median_immunity_wins_per_day": "{:.3f}"
        }
    )
)

print("Overall MBTI-known winner rate:", round(overall_winner_rate, 3))
print("Median MBTI type count:", type_count_median)

display(
    underrepresented_winners.style.format(
        {
            "winner_rate": "{:.3f}",
            "representation_share": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "median_days_lasted": "{:.1f}",
            "median_votes_per_day": "{:.3f}",
            "median_immunity_wins_per_day": "{:.3f}"
        }
    )
)

# Visual: winner rate by MBTI type
plot_df = mbti_type_summary.sort_values("winner_rate", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=plot_df,
    x="mbti_type",
    y="winner_rate",
    hue="representation_group",
    errorbar=None,
    ax=ax
)

ax.axhline(
    overall_winner_rate,
    linestyle="--",
    linewidth=1,
    label="Overall winner rate"
)

ax.set_title("Winner Rate by MBTI Type")
ax.set_xlabel("MBTI Type")
ax.set_ylabel("Winner Rate")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Representation")

plt.tight_layout()
plt.show()

# Text
if len(underrepresented_winners) > 0:
    top_underrepresented = underrepresented_winners.iloc[0]

    display(Markdown(
        f"**Answer:** The strongest underrepresented type by winner rate is "
        f"**{top_underrepresented['mbti_type']}**, with "
        f"**{top_underrepresented['winners']} winners out of "
        f"{top_underrepresented['players']} players**. Its winner rate is "
        f"**{top_underrepresented['winner_rate']:.3f}**, compared with the "
        f"overall MBTI-known winner rate of **{overall_winner_rate:.3f}**. "
        f"This is interesting, but it should be read carefully because rare "
        f"types can swing heavily when one person wins."
    ))
else:
    display(Markdown(
        "**Answer:** No underrepresented MBTI type clearly rises above the "
        "overall winner rate in this version of the cleaned data. That supports "
        "the earlier heatmap result: MBTI may be interesting, but it is not a "
        "strong standalone explanation for winning."
    ))

## Question 2: Are rare MBTI types different from common MBTI types?
The first question looks at individual MBTI types, but those groups can be tiny. This question zooms out and compares rare types against common types as groups. That makes the analysis more stable and helps test whether rarity itself is connected to winning, tenure, or vote exposure.

In [ ]:
# Question 2: Rare vs common MBTI types

rare_common_lookup = mbti_type_summary[
    ["mbti_type", "representation_group"]
].copy()

rare_common_df = mbti_phase3_df.merge(
    rare_common_lookup,
    left_on="personality_type_clean",
    right_on="mbti_type",
    how="left"
)

rare_common_summary = (
    rare_common_df
    .groupby("representation_group", observed=True)
    .agg(
        players=("player_name", "count"),
        winners=("is_winner_bool", "sum"),
        mbti_types=("personality_type_clean", "nunique"),
        median_survival_pct=("survival_pct", "median"),
        median_days_lasted=("days_lasted", "median"),
        median_votes_per_day=("votes_per_day", "median"),
        median_idols_per_day=("idols_per_day", "median"),
        median_immunity_wins_per_day=("immunity_wins_per_day", "median"),
        median_reward_wins_per_day=("reward_wins_per_day", "median")
    )
    .reset_index()
)

rare_common_summary["winner_rate"] = (
    rare_common_summary["winners"] / rare_common_summary["players"]
)

display(
    rare_common_summary.style.format(
        {
            "winner_rate": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "median_days_lasted": "{:.1f}",
            "median_votes_per_day": "{:.3f}",
            "median_idols_per_day": "{:.3f}",
            "median_immunity_wins_per_day": "{:.3f}",
            "median_reward_wins_per_day": "{:.3f}"
        }
    )
)

# Visual: rare vs common comparison
rare_common_plot_df = rare_common_summary.melt(
    id_vars="representation_group",
    value_vars=[
        "winner_rate",
        "median_survival_pct",
        "median_votes_per_day",
        "median_immunity_wins_per_day"
    ],
    var_name="metric",
    value_name="value"
)

fig, ax = plt.subplots(figsize=(11, 6))

sns.barplot(
    data=rare_common_plot_df,
    x="metric",
    y="value",
    hue="representation_group",
    errorbar=None,
    ax=ax
)

ax.set_title("Rare vs Common MBTI Groups")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Representation")

plt.tight_layout()
plt.show()

# Text
rare_common_sorted = rare_common_summary.sort_values(
    "winner_rate",
    ascending=False
)

top_group = rare_common_sorted.iloc[0]
bottom_group = rare_common_sorted.iloc[-1]

display(Markdown(
    f"**Answer:** The **{top_group['representation_group']}** group has the "
    f"higher winner rate in this comparison at **{top_group['winner_rate']:.3f}**, "
    f"compared with **{bottom_group['winner_rate']:.3f}** for the "
    f"**{bottom_group['representation_group']}** group. The important caveat is "
    f"that this does not prove rare personalities are better at Survivor. It only "
    f"shows that representation and winner rate are worth separating instead of "
    f"treating all MBTI types as equally sized groups."
))

## Question 3: Which eras do different MBTI types win more often in?

Survivor changes over time. Strategy, idols, casting, tribe swaps, and endgame expectations are not the same across every era. If MBTI has any useful signal, it may show up differently by era instead of appearing as one simple all-time pattern.

In [ ]:
# Question 3: MBTI winner patterns by era

era_type_summary = (
    mbti_phase3_df
    .dropna(subset=["era"])
    .groupby(["era", "personality_type_clean"], observed=True)
    .agg(
        players=("player_name", "count"),
        winners=("is_winner_bool", "sum"),
        median_survival_pct=("survival_pct", "median"),
        median_votes_per_day=("votes_per_day", "median")
    )
    .reset_index()
    .rename(columns={"personality_type_clean": "mbti_type"})
)

era_type_summary["winner_rate"] = (
    era_type_summary["winners"] / era_type_summary["players"]
)

display(
    era_type_summary
    .sort_values(["era", "winner_rate", "players"], ascending=[True, False, False])
    .style.format(
        {
            "winner_rate": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "median_votes_per_day": "{:.3f}"
        }
    )
)

# Winner counts by era and type are more stable than rates for tiny groups
era_winner_counts = (
    era_type_summary
    .pivot_table(
        index="era",
        columns="mbti_type",
        values="winners",
        aggfunc="sum",
        fill_value=0,
        observed=True
    )
)

display(era_winner_counts)

fig, ax = plt.subplots(figsize=(13, 6))

sns.heatmap(
    era_winner_counts,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    linewidths=0.5,
    ax=ax
)

ax.set_title("Number of Winners by MBTI Type and Era")
ax.set_xlabel("MBTI Type")
ax.set_ylabel("Era")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

# Best type per era by winner count, using winner rate as a tie-breaker
best_era_types = (
    era_type_summary
    .sort_values(
        ["era", "winners", "winner_rate", "players"],
        ascending=[True, False, False, False]
    )
    .groupby("era", observed=True)
    .head(1)
    .reset_index(drop=True)
)

display(
    best_era_types[
        [
            "era",
            "mbti_type",
            "players",
            "winners",
            "winner_rate",
            "median_survival_pct"
        ]
    ].style.format(
        {
            "winner_rate": "{:.3f}",
            "median_survival_pct": "{:.2f}"
        }
    )
)

era_sentences = []

for _, row in best_era_types.iterrows():
    era_sentences.append(
        f"- In **{row['era']}**, the leading winner-count MBTI type is "
        f"**{row['mbti_type']}** with **{int(row['winners'])} winners** "
        f"out of **{int(row['players'])} players**."
    )

display(Markdown(
    "**Answer:** The winning MBTI pattern changes by era rather than forming "
    "one clean all-time personality trend.\n\n" + "\n".join(era_sentences)
))

## Question 4: Do certain MBTI types have longer tenure than others?

Winning is rare, so winner rate alone can be noisy. Tenure gives a broader view of performance because it asks who tends to last longer, not only who wins. This question checks whether MBTI types or MBTI letter dimensions show meaningful differences in survival depth.

In [ ]:
# Question 4: MBTI and tenure

mbti_tenure_summary = (
    mbti_phase3_df
    .groupby("personality_type_clean", observed=True)
    .agg(
        players=("player_name", "count"),
        winners=("is_winner_bool", "sum"),
        median_survival_pct=("survival_pct", "median"),
        mean_survival_pct=("survival_pct", "mean"),
        median_days_lasted=("days_lasted", "median"),
        mean_days_lasted=("days_lasted", "mean"),
        median_finish_order=("finish_order", "median"),
        median_votes_per_day=("votes_per_day", "median")
    )
    .reset_index()
    .rename(columns={"personality_type_clean": "mbti_type"})
)

mbti_tenure_summary["winner_rate"] = (
    mbti_tenure_summary["winners"] / mbti_tenure_summary["players"]
)

display(
    mbti_tenure_summary
    .sort_values("median_survival_pct", ascending=False)
    .style.format(
        {
            "winner_rate": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "mean_survival_pct": "{:.2f}",
            "median_days_lasted": "{:.1f}",
            "mean_days_lasted": "{:.1f}",
            "median_finish_order": "{:.1f}",
            "median_votes_per_day": "{:.3f}"
        }
    )
)

# Letter-level tenure is more stable than full MBTI type tenure
dimension_columns = {
    "E/I": "E_or_I",
    "S/N": "S_or_N",
    "T/F": "T_or_F",
    "J/P": "J_or_P"
}

letter_tenure_frames = []

for dimension_name, column_name in dimension_columns.items():
    temp = (
        mbti_phase3_df
        .groupby(column_name, observed=True)
        .agg(
            players=("player_name", "count"),
            winners=("is_winner_bool", "sum"),
            median_survival_pct=("survival_pct", "median"),
            median_days_lasted=("days_lasted", "median"),
            median_votes_per_day=("votes_per_day", "median")
        )
        .reset_index()
        .rename(columns={column_name: "letter"})
    )

    temp["dimension"] = dimension_name
    temp["winner_rate"] = temp["winners"] / temp["players"]

    letter_tenure_frames.append(temp)

letter_tenure_summary = pd.concat(letter_tenure_frames, ignore_index=True)

display(
    letter_tenure_summary[
        [
            "dimension",
            "letter",
            "players",
            "winners",
            "winner_rate",
            "median_survival_pct",
            "median_days_lasted",
            "median_votes_per_day"
        ]
    ].style.format(
        {
            "winner_rate": "{:.3f}",
            "median_survival_pct": "{:.2f}",
            "median_days_lasted": "{:.1f}",
            "median_votes_per_day": "{:.3f}"
        }
    )
)

# Visual: full MBTI tenure
fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=mbti_tenure_summary.sort_values(
        "median_survival_pct",
        ascending=False
    ),
    x="mbti_type",
    y="median_survival_pct",
    errorbar=None,
    ax=ax
)

ax.set_title("Median Survival Depth by MBTI Type")
ax.set_xlabel("MBTI Type")
ax.set_ylabel("Median Survival Percent")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Visual w/ letter-level tenure text
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=letter_tenure_summary,
    x="dimension",
    y="median_survival_pct",
    hue="letter",
    errorbar=None,
    ax=ax
)

ax.set_title("Median Survival Depth by MBTI Letter Dimension")
ax.set_xlabel("MBTI Dimension")
ax.set_ylabel("Median Survival Percent")
ax.legend(title="Letter")

plt.tight_layout()
plt.show()

top_tenure_type = mbti_tenure_summary.sort_values(
    ["median_survival_pct", "players"],
    ascending=[False, False]
).iloc[0]

display(Markdown(
    f"**Answer:** The MBTI type with the highest median survival depth is "
    f"**{top_tenure_type['mbti_type']}**, with a median survival percent of "
    f"**{top_tenure_type['median_survival_pct']:.2f}** across "
    f"**{int(top_tenure_type['players'])} players**. This is a better signal "
    f"than winner rate alone because it looks at how long players stay in the "
    f"game, not only whether they reach the final outcome."
))

# Conclusion and Reflection

## Summary

This Phase 3 section started from a simple question: if the MBTI heatmap showed weak correlations, is there still a better personality trend hiding underneath the surface?

The answer is yes, but only in a limited and careful way. MBTI does not strongly explain who wins Survivor by itself. Some personality types may appear underrepresented while still having a higher winner rate, but those groups are small enough that one winner can shift the result. That means the pattern is interesting, but not strong enough to treat as a prediction.

The rare versus common type comparison gives a cleaner view. Instead of only looking at individual MBTI types, it groups types by representation and compares winner rate, survival depth, and vote exposure. This helps show whether rarity itself matters, or whether the rare-type trend is mostly a sample size issue.

The era analysis also matters because Survivor changes over time. A personality type that succeeds in one era may not show the same pattern in another era. That makes sense because the game itself changed through idols, swaps, advantages, casting style, and jury expectations.

The tenure analysis is probably the most stable part of this MBTI section. Winning is rare, but lasting longer gives us a broader view of performance. If certain MBTI types or letter dimensions survive deeper into the game, that is a more useful pattern than only asking who won.

## Limitations

The biggest limitation is sample size. Full MBTI types split the data into 16 groups, and some of those groups are small. Because of that, winner rate can look dramatic even when it is based on only a few players.

MBTI should also be treated carefully. It is not an exact measurement here, and it should not be used to make hard claims about personality or ability. It is only an exploratory category in the dataset.

There is also survivorship bias. Players who last longer naturally have more chances to receive votes, find idols, and win challenges. I used survival percent and per-day metrics to reduce that problem, but those adjustments do not fully remove it.

Finally, this analysis is observational. It can show relationships and differences, but it cannot prove that a personality type caused a player to win or last longer. Survivor is a social game, and many important factors are not captured in the CSV files, such as alliances, jury perception, confessionals, threat level, and private strategy.

## Reflection and Future Work

This project reminded me that a weak correlation is not the same thing as a dead question. The first MBTI heatmap did not show a strong direct relationship, but it still opened the door to better questions about rarity, representation, era, and tenure.

The biggest improvement was moving from "Does MBTI predict winning?" to "Where does MBTI still show small patterns?" and "Where does the data warn us not to overclaim?" That feels like a more holistic analysis.

For future work, I would compare MBTI only among deep runners or finalists. That would remove some of the noise from early boots and make the comparison more fair. I would also connect this with jury vote data, because Survivor is ultimately decided by people, not just by stats. A stronger future version could ask whether certain personality types are more likely to reach the end, receive jury votes, or lose as finalists.

Overall, MBTI is not the main answer to Survivor success. It is just one perspective. The better understanding is that winning comes from timing, social positioning, survival depth, and managing how other players perceive you. Personality may shape that path, but it does not dictate the path by itself.

# Phase 4 Machine Learning

## The Prediction

### Overview
We will be predicting the winner within a season, using the column `is_winner` as the target. Since this is a binary classification, the method used will be a random forest, which is more resistant to overfitting than decision trees.

The features used at a high level are gameplay columns (challenge wins and idols), social exposure (votes), the year of the season, and the MBTI (personality).

**Feature List:** `immunity_challenge_wins`, `reward_challenge_wins`, `immunity_idols_obtained`, `votes_per_day`, `season_year`, `personality_type`

The following features will be excluded from the model: `day`, `order`, `full_name`, `season`. `day` and `order` are tautological, since the objective of the game is to last the longest. Contestant and season names do not provide any useful data other than labeling.

### Connections to Earlier Phases
From Phase 2, the predictions from the model should link to earlier findings in our Outplay by Era visual and help determine which stats are most important to the winner, and help highlight findings from the votes per day analysis and MBTI heatmaps.

From Phase 3, we will be able to see more clearly how depth normalization affected idols, votes, and personality speculation.

### Goal
The model should be able to separate winners from non-winners using the provided features that have been collected at the end of the game. Since the model runs on data collected throughout the game, it cannot forecast who the winner will eventually be. Essentially, we are answering:

Given the end-game metrics for each contestant within a season, which profiles best match winners vs. non-winners.

### Limitations
There is an inherent classification imbalance, as only about 5% of the contestants are winners. Evaluation will need to rely on precision and recall more than accuracy.

## Model Implementation - Emma

Random Forest Classifier
- X/y
- split
- encode
- model fit

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report


# Use depth_df to create model (remove time related bias)
ml_df = depth_df.copy()

#encode personality type
le = LabelEncoder()
ml_df["personality_type"] = le.fit_transform(ml_df['personality_type'].astype(str))


# Select the target and features
y = ml_df["is_winner"]
X = ml_df[["immunity_wins_per_day", "reward_wins_per_day", "idols_per_day", "votes_per_day", "season_year", "personality_type"]]

# Keep only the columns in X and y
ml_df = ml_df[X.columns.tolist() + [y.name]]

# check dataframe
X.info()
y.value_counts()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    oob_score=True,
    random_state=42,
    class_weight="balanced" #this accounts for there being much less winners than losers
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

model.oob_score_

cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Winner", "Winner"]
))



### Tuning The Model
Since the model is getting 95% accuracy even though it only predicted 1/9 winners, we will attempt to tune it to predict winners successfully. Since we did not experience this problem in our class examples, I consulted Claude which suggested lowering the prediction threshold. In our first model the default threshold is 0.5, meaning the model should only predict "winner" if its at least 50% confident. We will test lowering the threshold to 5%-30% to try to increase winner prediction while keeping false positives low. 

In [ ]:
#get probabilities instead of hard predictions
y_prob = model.predict_proba(X_test)[:,1]

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30]:
    y_pred_thresh = (y_prob >= threshold).astype(int)
    print(f"\n----------Threshold: {threshold}------------------")
    print(confusion_matrix(y_test, y_pred_thresh))
    print(classification_report(y_test, y_pred_thresh, target_names=["Not Winner", "Winner"]))

A threshold of 0.2 has the best performance of 4/9 winners predicted with only three false positives. Let's look closer at the winners the model missed to see how confident the model was about them being non-winners.

In [ ]:
missed_winners = y_prob[(y_test == 1) & ((y_prob >= 0.20).astype(int) == 0)]
print(missed_winners)

Here is a better way of visualizing this: 
| Actual Winner | Model's Confidence They'd Win |
|---------------|-------------------------------|
| Winner 1      | 8%                            |
| Winner 2      | 9%                            |
| Winner 3      | 10%                           |
| Winner 4      | 3%                            |
| Winner 5      | 5%                            |

For all of these players the model is very confident they would not win based off of the features, so tweaking the model would not improve this. 

### Final Visualization: Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_prob = model.predict_proba(X_test)[:,1]
y_pred_thresh = (y_prob >= 0.2).astype(int)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_thresh,
    display_labels=["Not Winner", "Winner"],
    cmap="Blues"
)
plt.title("Confusion Matrix - Survivor Winner Prediction (Threshold 0.20)")

## Evaluation - Jarren
- Metrics on test set
- Confusion matrix
- Classification Report

## Interpretation
- Result Description of
- Importances
- Out-of-Bag (OOB)
- Over/under fit

## AI disclaimer

**Tool:** Cursor, Claude 

**Used for:** syntax questions, grammar, review, structure

**Scope:** refinement